# Exploring your own dataset with R

This notebook uses **R** to load a dataset *you* upload, look at it, and make some plots.
It's built for time-series data (anything with a **date** column and a **number** column, like snow depth, temperature, or stream flow), but most of it works for any table.

----

To 'run' a cell, hit the ▶ play button next to each code block (or press **Shift + Enter**).

Once a cell has run you'll see a number like [1] and a green check mark ✔ appear to the left of it. Try it on the next cell.

> **First time?** Check the runtime says **R**: go to *Runtime → Change runtime type* and pick **R**. (This notebook asks for R automatically, so it should already be set.)

In [ ]:
# Load the libraries (packages) we'll use.
# Most are already installed on Colab; anything missing gets installed (this can take a minute).
pkgs <- c("readr", "readxl", "dplyr", "ggplot2", "lubridate")
missing <- pkgs[!sapply(pkgs, requireNamespace, quietly = TRUE)]
if (length(missing) > 0) install.packages(missing)

suppressPackageStartupMessages({
  library(readr)
  library(readxl)
  library(dplyr)
  library(ggplot2)
  library(lubridate)
})

options(repr.plot.width = 11, repr.plot.height = 5)   # make plots wide
cat("Ready to go!\n")

-----
-----

# A) Upload your data and take a look

### How to upload a file
1. Click the **folder icon** 📁 in the left sidebar.
2. Click the **upload icon** (page with an up arrow) and choose your file — **.csv**, **.tsv**, **.txt**, or Excel (**.xlsx / .xls**).
3. Wait for it to appear in the file list, then run the cell below.

The cell finds the file you uploaded most recently. If you uploaded more than one, type the name you want between the quotes on the `data_file` line.

*No file yet? The cell will load a built-in example (monthly air temperatures in Nottingham, England, 1920–1939) so you can still try everything.*

⚠️ Uploaded files disappear when the Colab session ends, so you'll need to upload again next time.

In [ ]:
# 1. Find and read your file
data_file <- ""   # <-- optional: type a file name here, e.g. "my_data.csv"

upload_dir <- if (dir.exists("/content")) "/content" else getwd()

if (data_file == "") {
  candidates <- list.files(upload_dir, pattern = "\\.(csv|tsv|txt|xlsx|xls)$",
                           ignore.case = TRUE, full.names = TRUE)
  if (length(candidates) > 0) data_file <- candidates[which.max(file.mtime(candidates))]
} else if (!file.exists(data_file)) {
  data_file <- file.path(upload_dir, data_file)
}

read_any <- function(path) {
  ext <- tolower(tools::file_ext(path))
  if (ext %in% c("xlsx", "xls")) {
    read_excel(path)
  } else if (ext == "tsv") {
    read_tsv(path, comment = "#", show_col_types = FALSE)
  } else if (ext == "csv") {
    # comment = "#" skips header notes like the ones in SNOTEL downloads
    read_csv(path, comment = "#", show_col_types = FALSE)
  } else {
    read_delim(path, delim = NULL, comment = "#", show_col_types = FALSE)  # guesses the separator
  }
}

if (data_file != "" && file.exists(data_file)) {
  raw <- read_any(data_file)
  cat("Loaded:", basename(data_file), "\n")
} else {
  raw <- tibble(Date = seq(as.Date("1920-01-01"), by = "month", length.out = length(nottem)),
                Temp_F = as.numeric(nottem))
  cat("No uploaded file found, so using the built-in example dataset.\n")
}

In [ ]:
# 2. Now that you have your data, let's look at how it's formatted
cat("Rows:", nrow(raw), "  Columns:", ncol(raw), "\n\n")
glimpse(raw)
head(raw, 10)

In [ ]:
# 3. A quick summary of every column (min, max, mean, missing values...)
summary(raw)

### Look:
- How many rows are in your dataset?
- What variables (columns) are included? What units are they in?
- Are there missing values (`NA`)? Where might they come from?

-----

## Pick the columns to plot

The next cell guesses which column holds the **date** and which holds the **value** to plot.
If it guesses wrong, type the column names (exactly as they appear above) between the quotes and run it again.

In [ ]:
# 4. Choose your date and value columns
date_col  <- ""   # <-- e.g. "Date"
value_col <- ""   # <-- e.g. "SWE_in"

cols <- names(raw)
if (date_col == "") {
  is_date <- sapply(raw, function(x) inherits(x, c("Date", "POSIXt")))
  by_name <- grepl("date|time|day", cols, ignore.case = TRUE)
  date_col <- if (any(is_date)) cols[is_date][1] else if (any(by_name)) cols[by_name][1] else cols[1]
}
if (value_col == "") {
  numeric_cols <- cols[sapply(raw, is.numeric) & cols != date_col]
  value_col <- if (length(numeric_cols) > 0) numeric_cols[1] else cols[cols != date_col][1]
}
stopifnot("date_col isn't a column name - check spelling"  = date_col  %in% cols,
          "value_col isn't a column name - check spelling" = value_col %in% cols)

# Turn the date column into real dates (handles 2024-01-31, 1/31/2024, 31-01-2024, etc.)
to_date <- function(x) {
  if (inherits(x, c("Date", "POSIXt"))) return(as.Date(x))
  as.Date(parse_date_time(as.character(x),
                          orders = c("ymd", "mdy", "dmy", "ymd HMS", "ymd HM", "mdy HMS", "mdy HM", "ym", "Y"),
                          quiet = TRUE))
}
to_number <- function(x) if (is.numeric(x)) x else suppressWarnings(parse_number(as.character(x)))

df <- tibble(date = to_date(raw[[date_col]]), value = to_number(raw[[value_col]])) %>%
  filter(!is.na(date), !is.na(value)) %>%
  arrange(date)

if (nrow(df) == 0) stop("No rows have both a readable date and a number. ",
                        "Check that date_col and value_col point to the right columns.")

cat("Date column: ", date_col, "\nValue column:", value_col, "\n")
cat("Usable rows: ", nrow(df), "of", nrow(raw), "\n")
cat("Date range:  ", format(min(df$date)), "to", format(max(df$date)), "\n")

In [ ]:
# 5. Make a quick plot of the dataset
ggplot(df, aes(x = date, y = value)) +
  geom_line(color = "steelblue") +
  labs(title = paste(value_col, "over time"), x = "Date", y = value_col) +
  theme_minimal(base_size = 14)

### Look:
- What do you notice about these data? Is there a repeating pattern?
- When were the highest values? How high did they get?
- When were the lowest values?
- Do you see gaps or odd spikes that might be errors?

----------
---------

# B) Compare years against each other

Lining every year up on the same x-axis makes it easy to see which years were unusual.

Some fields use a shifted calendar. Water managers use a **water year** that starts on **October 1** (so October 2023 belongs to water year 2024).
Set `start_month` below: `1` for a normal calendar year, `10` for a water year.

*Why would a water year be useful for thinking about snow accumulation and melt?*

In [ ]:
# 1. Label every row with its year and how far into that year it is
start_month <- 1   # <-- 1 = calendar year (Jan 1), 10 = water year (Oct 1)

shift <- if (start_month > 1) 1 else 0
df <- df %>%
  mutate(
    year          = year(date) + ifelse(shift == 1 & month(date) >= start_month, 1, 0),
    year_start    = make_date(year - shift, start_month, 1),
    day_of_year   = as.numeric(date - year_start) + 1
  )

head(df, 10)

### Look:
- What new columns are in the data now?
- Check a few rows: does `day_of_year` make sense for the date?

In [ ]:
# 2. Plot every year on the same axis, with the most recent year highlighted
latest <- max(df$year)

ggplot(df, aes(x = day_of_year, y = value, group = year)) +
  geom_line(aes(color = year), alpha = 0.6) +
  geom_line(data = filter(df, year == latest), color = "black", linewidth = 1.3) +
  scale_color_viridis_c(name = "Year") +
  labs(title = paste(value_col, "by year (black line =", latest, ")"),
       x = paste("Day of year (starting", month.name[start_month], "1)"),
       y = value_col) +
  theme_minimal(base_size = 14)

In [ ]:
# 3. A table that summarizes each year
yearly <- df %>%
  group_by(year) %>%
  summarise(
    n_obs        = n(),
    mean_value   = round(mean(value), 2),
    max_value    = max(value),
    date_of_max  = date[which.max(value)],
    min_value    = min(value),
    .groups = "drop"
  )
yearly

In [ ]:
# 4. Which years had the highest peak?
ggplot(yearly, aes(x = factor(year), y = max_value)) +
  geom_col(fill = "steelblue") +
  geom_hline(yintercept = mean(yearly$max_value), linetype = "dashed") +
  labs(title = paste("Highest", value_col, "each year (dashed line = average)"),
       x = "Year", y = value_col) +
  theme_minimal(base_size = 14) +
  theme(axis.text.x = element_text(angle = 90, vjust = 0.5))

### Look:
- How is the year-by-year plot different from the first one?
- What new information can you see from it?
- Which years stand out? Check `n_obs`: a year with few observations may be incomplete.
- What other information would help you understand these data?

*See if you can start using AI to help you make new plots. Try asking for R code with `ggplot2`, and tell it your data frame is called `df` with columns `date`, `value`, `year`, and `day_of_year`.*

In [ ]:
# 5. (Optional) Save your yearly summary, then download it from the 📁 file panel
write_csv(yearly, "yearly_summary.csv")
cat("Saved yearly_summary.csv\n")

Try your own code here: